# Velocity Alignment Test

Quick sanity-check that ego velocity and box detection velocities live in the **same global frame** and are consistently aligned in sign and magnitude.

Both `EgoStateSE3.dynamic_state_se3.velocity_3d` and `BoxDetectionSE3.velocity_3d` are stored as `Vector3D` in the **global frame**. If everything is consistent:

- Ego arrow should point along the visible ego trajectory tangent.
- Parked objects should have ~zero arrows.
- Moving traffic should have arrows with plausible directions, independent of ego motion.
- Ego speed should match a finite-difference of consecutive ego positions.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Polygon as MplPolygon

from py123d.api import SceneAPI, SceneFilter, get_filtered_scenes

scene_filter = SceneFilter(
    datasets=["wod-perception"],
    split_names=None,
    log_names=None,
    scene_uuids=None,
    target_iteration_duration_s=0.1,
    future_duration_s=1.0,
    history_duration_s=0.5,
    required_scene_modalities=["ego_state_se3", "box_detections_se3"],
    shuffle=True,
)
scenes = get_filtered_scenes(scene_filter)
print(f"Found {len(scenes)} scenes")

In [ ]:
# Scale velocity arrows to a ~0.5 s lookahead, so the arrow tip shows where
# the object would be in 0.5 s if it kept moving at its current velocity.
VEL_LOOKAHEAD_S = 1
PLOT_RADIUS_M = 60.0


def _draw_polygon(ax, polygon, **kwargs):
    coords = np.asarray(polygon.exterior.coords)
    ax.add_patch(MplPolygon(coords[:, :2], closed=True, **kwargs))


def _draw_velocity(ax, x, y, vx, vy, color):
    dx, dy = vx * VEL_LOOKAHEAD_S, vy * VEL_LOOKAHEAD_S
    if np.hypot(dx, dy) < 1e-3:
        return
    ax.arrow(
        x,
        y,
        dx,
        dy,
        head_width=0.6,
        head_length=0.8,
        length_includes_head=True,
        fc=color,
        ec=color,
        linewidth=1.2,
    )


def _draw_velocity_components(ax, x, y, vx, vy, color_x, color_y):
    """Draw vx and vy as two separate arrows from the same origin."""
    dx = vx * VEL_LOOKAHEAD_S
    dy = vy * VEL_LOOKAHEAD_S
    if abs(dx) > 1e-3:
        ax.arrow(
            x,
            y,
            dx,
            0,
            head_width=0.4,
            head_length=0.5,
            length_includes_head=True,
            fc=color_x,
            ec=color_x,
            linewidth=1.2,
            alpha=0.8,
        )
    if abs(dy) > 1e-3:
        ax.arrow(
            x,
            y,
            0,
            dy,
            head_width=0.4,
            head_length=0.5,
            length_includes_head=True,
            fc=color_y,
            ec=color_y,
            linewidth=1.2,
            alpha=0.8,
        )


def _setup_bev_ax(ax, cx, cy, title):
    ax.set_aspect("equal")
    ax.set_xlim(cx - PLOT_RADIUS_M, cx + PLOT_RADIUS_M)
    ax.set_ylim(cy - PLOT_RADIUS_M, cy + PLOT_RADIUS_M)
    ax.set_xlabel("global x [m]")
    ax.set_ylabel("global y [m]")
    ax.set_title(title)
    ax.grid(True, alpha=0.3)


def _get_frame_data(scene: SceneAPI, iteration: int):
    ego = scene.get_ego_state_se3_at_iteration(iteration=iteration)
    boxes = scene.get_box_detections_se3_at_iteration(iteration=iteration)
    assert ego is not None and boxes is not None

    boxes = boxes.box_detections if boxes.box_detections is not None else []
    boxes += [ego.box_detection_se3]

    return ego, boxes


def plot_bev(scene: SceneAPI, iteration: int, ax) -> None:
    """Draw a top-down BEV of ego + box detections with velocity arrows, all in the global frame."""
    ego, boxes = _get_frame_data(scene, iteration)
    ego_xyz = ego.center_se3.point_3d
    cx, cy = float(ego_xyz.x), float(ego_xyz.y)

    # Box detections + velocities
    for box in boxes:
        _draw_polygon(ax, box.shapely_polygon, facecolor="tab:blue", edgecolor="navy", alpha=0.35)
        if box.velocity_3d is not None:
            bx = float(box.bounding_box_se3.center_se3.point_3d.x)
            by = float(box.bounding_box_se3.center_se3.point_3d.y)
            _draw_velocity(ax, bx, by, float(box.velocity_3d.x), float(box.velocity_3d.y), color="navy")

    _setup_bev_ax(ax, cx, cy, f"iteration={iteration}")


def plot_bev_components(scene: SceneAPI, iteration: int, ax) -> None:
    """Like plot_bev, but draws vx (red/orange) and vy (green/teal) as separate arrows."""
    ego, boxes = _get_frame_data(scene, iteration)
    ego_xyz = ego.center_se3.point_3d
    cx, cy = float(ego_xyz.x), float(ego_xyz.y)

    # Box detections + component arrows
    for box in boxes:
        _draw_polygon(ax, box.shapely_polygon, facecolor="tab:blue", edgecolor="navy", alpha=0.35)
        if box.velocity_3d is not None:
            bx = float(box.bounding_box_se3.center_se3.point_3d.x)
            by = float(box.bounding_box_se3.center_se3.point_3d.y)
            _draw_velocity_components(
                ax, bx, by, float(box.velocity_3d.x), float(box.velocity_3d.y), color_x="tab:orange", color_y="teal"
            )

    _setup_bev_ax(ax, cx, cy, f"iteration={iteration} (vx/vy components)")

## Single frame

Check that the red ego arrow originates at the ego rectangle center and points forward along the ego heading, and that box arrows look plausible.

In [ ]:
scene: SceneAPI = np.random.choice(scenes)  # type: ignore
print(f"Using scene from dataset={scene.dataset}, log={scene.log_name}")

fig, ax = plt.subplots(figsize=(12, 12))
plot_bev(scene, iteration=0, ax=ax)
plt.show()

## Single frame — dx / dy components

Same frame, but vx and vy drawn as **separate** axis-aligned arrows:
- **Ego**: red = vx, green = vy
- **Boxes**: orange = vx, teal = vy

In [ ]:
fig, ax = plt.subplots(figsize=(12, 12))
plot_bev_components(scene, iteration=0, ax=ax)
plt.show()

## Multiple iterations

Across the scene horizon, the per-iteration ego arrow should consistently point along the direction of the ego's displacement between frames.

In [ ]:
meta = scene.scene_metadata
iterations = list(range(-meta.num_history_iterations, meta.num_future_iterations + 1, 3))

ncols = 3
nrows = int(np.ceil(len(iterations) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 5 * nrows), squeeze=False)
for ax, it in zip(axes.flat, iterations):
    plot_bev(scene, iteration=it, ax=ax)
for ax in axes.flat[len(iterations) :]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## Scalar check: ego velocity vs finite-difference speed

The norm of the stored ego velocity should match the finite-difference of consecutive ego positions within a few percent.

In [ ]:
dt = scene.scene_metadata.iteration_duration_s
iters = list(range(-meta.num_history_iterations, meta.num_future_iterations))

rows = []
for it in iters:
    ego_a = scene.get_ego_state_se3_at_iteration(iteration=it)
    ego_b = scene.get_ego_state_se3_at_iteration(iteration=it + 1)
    if ego_a is None or ego_b is None or ego_a.dynamic_state_se3 is None:
        continue
    pa = ego_a.center_se3.point_3d
    pb = ego_b.center_se3.point_3d
    fd = np.linalg.norm([float(pb.x) - float(pa.x), float(pb.y) - float(pa.y)]) / dt
    v = ego_a.dynamic_state_se3.velocity_3d
    stored = float(np.linalg.norm([float(v.x), float(v.y)]))
    rows.append((it, stored, fd, abs(stored - fd)))

print(f"{'iter':>5}  {'stored |v| [m/s]':>18}  {'finite-diff |v| [m/s]':>22}  {'|diff|':>8}")
for it, stored, fd, err in rows:
    print(f"{it:>5}  {stored:>18.3f}  {fd:>22.3f}  {err:>8.3f}")